# Explore a real ingested Discogs catalog

Real data, not a fixture. Reads the actual Parquet dataset a completed
`scripts/run-ingest.sh` run produced, mounted read-only into this container
and the local `dask-worker` containers at `/data/discogs` (see
`docker-compose.jupyter.yml` / `docker-stack.dask.yml`'s `DISCOGS_PROCESSED_DIR`
bind mount). Uses the exact same convention as
`scripts/profile-discogs-dataset.sh`: a `SNAPSHOT` env var selecting
`snapshot=<date>` under the mounted root.

**If you haven't run a real ingest yet**, the next cell fails with a clear
message -- see `docs/DISCOGS_INGESTION.md`. This notebook never silently
substitutes synthetic data and presents it as real.

**Known limitation:** if a second Dask worker is running on the optional
build node (`docker-compose.dask-worker-remote.yml`), it does NOT have this
mount unless that host separately has the same processed dataset --
partitions scheduled there will fail. See that file's own comment.


In [ ]:
import os
from pathlib import Path

PROCESSED_ROOT = Path("/data/discogs")
snapshot = os.environ.get("SNAPSHOT")
if not snapshot:
    raise RuntimeError(
        "Set SNAPSHOT (e.g. SNAPSHOT=20260601) as a Jupyter kernel/container "
        "env var, matching a completed scripts/run-ingest.sh run. See "
        "docs/DISCOGS_INGESTION.md."
    )

dataset_root = PROCESSED_ROOT / f"snapshot={snapshot}"
releases_glob = str(dataset_root / "table=releases" / "*.parquet")
credits_glob = str(dataset_root / "table=credits" / "*.parquet")

if not (dataset_root / "table=releases").is_dir():
    raise RuntimeError(
        f"No dataset at {dataset_root} -- confirm DISCOGS_PROCESSED_DIR in "
        "local/dask.env points at the right host path, and that "
        f"snapshot={snapshot} has actually completed (docs/DISCOGS_INGESTION.md)."
    )
print(f"Using real dataset at {dataset_root}")

In [ ]:
import duckdb

# Local, non-distributed profiling for comparison -- same pattern as
# scripts/profile-discogs-dataset.sh.
con = duckdb.connect()
con.execute(f"CREATE VIEW releases AS SELECT * FROM '{releases_glob}'")
con.execute(f"CREATE VIEW credits AS SELECT * FROM '{credits_glob}'")
con.sql("SELECT count(*) AS release_rows FROM releases").show()
con.sql("SELECT country, count(*) n FROM releases GROUP BY 1 ORDER BY 2 DESC LIMIT 10").show()

In [ ]:
import dask.dataframe as dd
from dask.distributed import Client

client = Client("tcp://dask-scheduler:8786")

# Real distributed read -- each worker opens the partitions it's assigned
# directly from the shared /data/discogs mount, no data movement through
# this notebook's own kernel. Real bug found and fixed against the actual
# completed ingest (3,839 real part files per table): dd.read_parquet()
# without an explicit blocksize aggregated ALL of them into a SINGLE
# ~3.9GB partition for credits (confirmed via .npartitions == 1 below) --
# a real distributed join then has to shuffle that whole table through one
# worker at once, which is what OOM-killed it. blocksize="64MB" forces
# many smaller partitions instead, matching how a real production job
# would actually want to read this dataset.
releases = dd.read_parquet(releases_glob, blocksize="64MB")
credits = dd.read_parquet(credits_glob, blocksize="64MB")
print(f"releases partitions: {releases.npartitions}, credits partitions: {credits.npartitions}")

In [ ]:
# A genuinely distributed computation: releases per country, then a
# releases<->credits join -- cross-check the count against the DuckDB view
# above as a correctness sanity check, not just a speed demo.
by_country = releases.groupby("country").size().compute().sort_values(ascending=False)
print(by_country.head(10))

joined_count = releases.merge(credits, on="release_id", how="inner").shape[0].compute()
duckdb_joined_count = con.sql(
    "SELECT count(*) FROM releases r JOIN credits c USING (release_id)"
).fetchone()[0]
assert joined_count == duckdb_joined_count, (joined_count, duckdb_joined_count)
print(f"releases-credits join row count (Dask, cross-checked against DuckDB): {joined_count}")